# Dataset - DataFrame

### 1. Preparar el entorno e importar la clase de carga
El notebook puede ejecutarse desde distintas carpetas. Por eso se buscan los directorios padres hasta encontrar src/clases_po y se agrega src a sys.path. Así se puede importar HFDatasetLoader sin instalar el proyecto como paquete.

In [1]:
import sys
import pandas as pd
from pathlib import Path
import numpy as np

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "clases_po").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from clases_po import HFDatasetLoader

/workspaces/Machine-Learning-/env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 2. Configurar el dataset y la salida
dataset_name identifica el repositorio en Hugging Face y split="train" indica la división que se descargará. El parámetro filename hace que el DataFrame también se guarde como CSV en src/carga_datos/datos/ cuando el notebook se ejecuta desde esa carpeta.

In [2]:
loader = HFDatasetLoader(
    dataset_name="sayanroy058/Business-Sales",
    split="train",
    filename="datos/Business-Sales.csv",
    
)

### 3. Descargar el dataset y crear el DataFrame
El método load() descarga el dataset, lo convierte a pandas.DataFrame y lo asigna a la variable df. La expresión df.head() muestra las primeras filas para comprobar que la carga fue exitosa.

In [ ]:
df_raw = loader.load()
df_raw.head()

,year,make,model,trim,body,transmission,vin,state,condition,odometer,color,interior,seller,mmr,sellingprice,saledate
0,2015,Kia,Sorento,LX,SUV,automatic,5xyktca69fg566472,ca,5.0,16639.0,white,black,kia motors america inc,20500.0,21500.0,Tue Dec 16 2014 12:30:00 GMT-0800 (PST)
1,2015,Kia,Sorento,LX,SUV,automatic,5xyktca69fg561319,ca,5.0,9393.0,white,beige,kia motors america inc,20800.0,21500.0,Tue Dec 16 2014 12:30:00 GMT-0800 (PST)
2,2014,BMW,3 Series,328i SULEV,Sedan,automatic,wba3c1c51ek116351,ca,45.0,1331.0,gray,black,financial services remarketing (lease),31900.0,30000.0,Thu Jan 15 2015 04:30:00 GMT-0800 (PST)
3,2015,Volvo,S60,T5,Sedan,automatic,yv1612tb4f1310987,ca,41.0,14282.0,white,black,volvo na rep/world omni,27500.0,27750.0,Thu Jan 29 2015 04:30:00 GMT-0800 (PST)
4,2014,BMW,6 Series Gran Coupe,650i,Sedan,automatic,wba6b2c57ed129731,ca,43.0,2641.0,gray,black,financial services remarketing (lease),66000.0,67000.0,Thu Dec 18 2014 12:30:00 GMT-0800 (PST)


: 

### 4. Limpieza de datos
Antes de limpiar se revisa la estructura original: 558.837 filas y 16 columnas. Cada fila es la venta de un vehículo usado en subasta, con sus características (año, marca, modelo, carrocería, millas recorridas, estado), el precio de referencia del mercado (`mmr`) y el precio final de venta (`sellingprice`).

### 5.Resumen de nulos 

#resumen_nulos = pd.DataFrame({
  #  'Total_Nulos': df.isnull().sum(),
 #   'Porcentaje_%': (df.isnull().mean() * 100).round(2)
#})

# Mostrar solo las columnas que contienen al menos un nulo
#print(resumen_nulos[resumen_nulos['Total_Nulos'] > 0])

In [ ]:
# Copia de trabajo para preservar el DataFrame original
df = df_raw.copy()

# Registramos métricas iniciales
dim_inicial = df.shape
nulos_iniciales = df.isnull().sum().to_dict()

# ---------------------------------------------------------
# Step 1: Depuración de registros corruptos y desalineados
# ---------------------------------------------------------
# Se eliminan filas donde 'year' no sea numérico o 'vin' tenga longitud inválida (< 13 caracteres)
df['year_numeric'] = pd.to_numeric(df['year'], errors='coerce')
invalid_rows_mask = df['year_numeric'].isna() | (df['vin'].astype(str).str.len() < 13)
df_cleaned_step1 = df[~invalid_rows_mask].copy()
df_cleaned_step1.drop(columns=['year_numeric'], inplace=True)
affected_corrupt = invalid_rows_mask.sum()

# ---------------------------------------------------------
# Step 2: Conversión y corrección de tipos de datos
# ---------------------------------------------------------
# Convertir 'year' a entero
df_cleaned_step1['year'] = df_cleaned_step1['year'].astype(int)

# Formatear 'saledate' eliminando sufijos de zona horaria compleja y convirtiendo a Datetime UTC
# Ejemplo de valor: "Tue Dec 16 2014 12:30:00 GMT-0800 (PST)"
cleaned_dates = df_cleaned_step1['saledate'].astype(str).str.replace(r'\s*GMT.*$', '', regex=True)
df_cleaned_step1['saledate'] = pd.to_datetime(cleaned_dates, errors='coerce', utc=True)

# Asegurar tipos numéricos continuos
numeric_cols = ['condition', 'odometer', 'mmr', 'sellingprice']
for col in numeric_cols:
    df_cleaned_step1[col] = pd.to_numeric(df_cleaned_step1[col], errors='coerce')



/tmp/ipykernel_12965/2303658506.py:12: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df['year_numeric'] = pd.to_numeric(df['year'], errors='coerce')
/tmp/ipykernel_12965/2303658506.py:22: FutureWarning: ChainedAssignmentError: behaviour will

In [ ]:
# ---------------------------------------------------------
# Step 3: Normalización de texto y categorías
# ---------------------------------------------------------
cat_cols = ['make', 'model', 'trim', 'body', 'transmission', 'state', 'color', 'interior', 'seller']

for col in cat_cols:
    df_cleaned_step1[col] = df_cleaned_step1[col].astype(str).str.strip().str.lower()

# Mapeo y corrección de sinónimos comunes en marcas y carrocerías
make_mapping = {
    'vw': 'volkswagen',
    'landrover': 'land rover',
    'gmc truck': 'gmc',
    'dodge-ram': 'dodge',
    'mercedes': 'mercedes-benz',
    'mercedes-b': 'mercedes-benz'
}
df_cleaned_step1['make'] = df_cleaned_step1['make'].replace(make_mapping)

body_mapping = {
    'sedan': 'sedan',
    'suv': 'suv',
    'coupe': 'coupe',
    'convertible': 'convertible',
    'hatchback': 'hatchback',
    'wagon': 'wagon',
    'supercrew': 'crew cab',
    'double cab': 'crew cab',
    'king cab': 'extended cab'
}
df_cleaned_step1['body'] = df_cleaned_step1['body'].replace(body_mapping)

# Normalizar estado a mayúsculas de 2 letras
df_cleaned_step1['state'] = df_cleaned_step1['state'].str.upper()

# ---------------------------------------------------------
# Step 4: Tratamiento de valores faltantes (NaN)
# ---------------------------------------------------------
# A. Variables Categóricas: Imputación estratégica con 'unknown'
text_impute_cols = ['make', 'model', 'trim', 'body', 'transmission', 'color', 'interior', 'seller']
for col in text_impute_cols:
    df_cleaned_step1[col] = df_cleaned_step1[col].replace(['nan', 'none', 'null', ''], np.nan)
    df_cleaned_step1[col] = df_cleaned_step1[col].fillna('unknown')

# B. Variables Numéricas: Imputación por mediana agrupada
# Mediana del odómetro según año y marca
df_cleaned_step1['odometer'] = df_cleaned_step1.groupby(['year', 'make'])['odometer'].transform(
    lambda x: x.fillna(x.median())
)
df_cleaned_step1['odometer'] = df_cleaned_step1['odometer'].fillna(df_cleaned_step1['odometer'].median())

# Mediana de condición
df_cleaned_step1['condition'] = df_cleaned_step1.groupby(['year'])['condition'].transform(
    lambda x: x.fillna(x.median())
)
df_cleaned_step1['condition'] = df_cleaned_step1['condition'].fillna(df_cleaned_step1['condition'].median())

# Filtrar registros marginales sin precio de venta o sin MMR (< 0.01% de los datos)
initial_rows_before_price_drop = len(df_cleaned_step1)
df_cleaned_step1 = df_cleaned_step1.dropna(subset=['sellingprice', 'mmr', 'saledate'])
affected_missing_prices = initial_rows_before_price_drop - len(df_cleaned_step1)

# ---------------------------------------------------------
# Step 5: Detección y eliminación de duplicados
# ---------------------------------------------------------
# Identificación de duplicados por VIN y fecha de venta exacta
initial_rows_before_dups = len(df_cleaned_step1)
df_prepared = df_cleaned_step1.drop_duplicates(subset=['vin', 'saledate'], keep='first').copy()
affected_duplicates = initial_rows_before_dups - len(df_prepared)

# ---------------------------------------------------------
# Step 6: Ingeniería de variables (Enriquecimiento)
# ---------------------------------------------------------
# Diferencia entre el precio de venta real y la estimación de mercado (MMR)
df_prepared['price_diff'] = df_prepared['sellingprice'] - df_prepared['mmr']
df_prepared['sale_year'] = df_prepared['saledate'].dt.year
df_prepared['sale_month'] = df_prepared['saledate'].dt.month

# Métricas finales
dim_final = df_prepared.shape
nulos_finales = df_prepared.isnull().sum().to_dict()